# Assignment 1 : FCNN




# Part 1: The "No-Framework" Challenge (NumPy Only)

## Project Overview

The goal of this project is to understand the inner workings of a neural network "engine" by building it from the ground up. We utilize the **UCI Adult Census Income** dataset to predict whether an individual's income exceeds $50K/yr based on census data.

## Question 1.1: Building the Engine

We implement a **3-layer Fully Connected Neural Network (FCNN)** using only **NumPy** and standard Python libraries.


## Question 1.2: The Importance of "Pre-Processing"

Neural networks are sensitive to the scale of input data. This section explores why normalization is a requirement, not a suggestion.


Cell 1: Installation & Imports

In [ ]:
!pip install ucimlrepo

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

EPSILON = 1e-15

Cell 2: Data Loading Utility

In [ ]:
def load_data():
    print("Fetching data from ucimlrepo...")
    adult = fetch_ucirepo(id=2)
    X = adult.data.features.copy()
    y = adult.data.targets.copy()

    
    df = pd.concat([X, y], axis=1).dropna()
    X_clean = df.iloc[:, :-1]
    y_clean = df.iloc[:, -1]

    # Target Encoding (>50K -> 1, <=50K -> 0)
    y_encoded = y_clean.astype(str).apply(lambda x: 1 if '>50K' in x else 0).values.reshape(-1, 1)

    # Feature Encoding (One-Hot)
    X_encoded = pd.get_dummies(X_clean, drop_first=True)

    # Train/Test Split (80/20)
    split = int(0.8 * len(X_encoded))
    X_train_raw = X_encoded.iloc[:split].values.astype(float)
    X_test_raw = X_encoded.iloc[split:].values.astype(float)

    # Min-Max Scaling
    min_val = X_train_raw.min(axis=0)
    max_val = X_train_raw.max(axis=0)
    denom = max_val - min_val
    denom[denom == 0] = 1

    X_train_scaled = (X_train_raw - min_val) / denom
    X_test_scaled = (X_test_raw - min_val) / denom

    return X_train_raw, X_test_raw, X_train_scaled, X_test_scaled, y_encoded[:split], y_encoded[split:]

Cell 3: The Neural Network Class

In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, h1, h2, output_size=1):
        # He-Initialization
        self.params = {
            'W1': np.random.randn(input_size, h1) * np.sqrt(2 / input_size),
            'b1': np.zeros((1, h1)),
            'W2': np.random.randn(h1, h2) * np.sqrt(2 / h1),
            'b2': np.zeros((1, h2)),
            'W3': np.random.randn(h2, output_size) * np.sqrt(2 / h2),
            'b3': np.zeros((1, output_size))
        }

        
        self.m = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.v = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.t = 0

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return (Z > 0).astype(float)

    def sigmoid(self, Z):
        # Clipping to prevent numerical overflow
        Z = np.clip(Z, -500, 500)
        return 1 / (1 + np.exp(-Z))

    def forward(self, X):
        Z1 = X @ self.params['W1'] + self.params['b1']
        A1 = self.relu(Z1)

        Z2 = A1 @ self.params['W2'] + self.params['b2']
        A2 = self.relu(Z2)

        Z3 = A2 @ self.params['W3'] + self.params['b3']
        A3 = self.sigmoid(Z3)

        cache = (X, Z1, A1, Z2, A2, Z3, A3)
        return A3, cache

    def compute_loss(self, A, Y):
        m = Y.shape[0]
        return -(1/m) * np.sum(Y*np.log(A+EPSILON) + (1-Y)*np.log(1-A+EPSILON))

    def backward(self, cache, Y):
        m = Y.shape[0]
        X, Z1, A1, Z2, A2, Z3, A3 = cache
        grads = {}

        dZ3 = A3 - Y
        grads['W3'] = A2.T @ dZ3 / m
        grads['b3'] = np.sum(dZ3, axis=0, keepdims=True) / m

        dA2 = dZ3 @ self.params['W3'].T
        dZ2 = dA2 * self.relu_derivative(Z2)
        grads['W2'] = A1.T @ dZ2 / m
        grads['b2'] = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = dZ2 @ self.params['W2'].T
        dZ1 = dA1 * self.relu_derivative(Z1)
        grads['W1'] = X.T @ dZ1 / m
        grads['b1'] = np.sum(dZ1, axis=0, keepdims=True) / m

        return grads

    def step_sgd(self, grads, lr):
        for k in self.params:
            self.params[k] -= lr * grads[k]

    def step_adam(self, grads, lr=0.001, beta1=0.9, beta2=0.999):
        self.t += 1
        for k in self.params:
            g = grads[k]
            self.m[k] = beta1*self.m[k] + (1-beta1)*g
            self.v[k] = beta2*self.v[k] + (1-beta2)*(g**2)

            m_hat = self.m[k] / (1 - beta1**self.t)
            v_hat = self.v[k] / (1 - beta2**self.t)

            self.params[k] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)

Cell 4: Training Function

In [4]:
def train(model, X, Y, epochs=50, batch_size=256, lr=0.01, optimizer="sgd"):
    n = X.shape[0]

    for epoch in range(epochs):
        # Shuffle
        idx = np.random.permutation(n)
        X = X[idx]
        Y = Y[idx]

        # Mini-batch training
        for i in range(0, n, batch_size):
            X_batch = X[i:i+batch_size]
            Y_batch = Y[i:i+batch_size]

            A, cache = model.forward(X_batch)
            grads = model.backward(cache, Y_batch)

            if optimizer == "sgd":
                model.step_sgd(grads, lr)
            else:
                model.step_adam(grads, lr)

        # Evaluate every 100 epochs
        if epoch % 100 == 0:
            A_full, _ = model.forward(X)
            loss = model.compute_loss(A_full, Y)
            acc = np.mean((A_full > 0.5) == Y)
            print(f"epoch {epoch} | loss {loss:.4f} | acc {acc:.4f}")

Cell 5: Execution (The Results)

In [ ]:
# Load Data
raw_tr, raw_te, scl_tr, scl_te, y_tr, y_te = load_data()


print("\n--- Question 1.1: Scaled Data with SGD ---")
model_scaled = NeuralNetwork(scl_tr.shape[1], 128, 64)

train(model_scaled, scl_tr, y_tr, epochs=1000, lr=0.1, optimizer="sgd")

acc_sgd = np.mean((model_scaled.forward(scl_te)[0] > 0.5) == y_te)
print(f"Scaled (SGD) test accuracy: {acc_sgd*100:.2f}%")



print("\n--- Question 1.2: Raw Data with SGD ---")
model_raw = NeuralNetwork(raw_tr.shape[1], 128, 64)

train(model_raw, raw_tr, y_tr, epochs=1000, lr=0.001, optimizer="sgd")

acc_sgd_raw = np.mean((model_raw.forward(raw_te)[0] > 0.5) == y_te)
print(f"Raw (SGD) test accuracy: {acc_sgd_raw*100:.2f}%")



print("\n--- Question 1.2: Raw Data with Adam ---")
model_raw = NeuralNetwork(raw_tr.shape[1], 128, 64)

train(model_raw, raw_tr, y_tr, epochs=1000, lr=0.001, optimizer="adam")

acc_adam_raw = np.mean((model_raw.forward(raw_te)[0] > 0.5) == y_te)
print(f"Raw (Adam) test accuracy: {acc_adam_raw*100:.2f}%")

Fetching data from ucimlrepo...

--- Question 1.1: Scaled Data with SGD ---
epoch 0 | loss 0.3739 | acc 0.8282
epoch 100 | loss 0.2888 | acc 0.8659
epoch 200 | loss 0.2646 | acc 0.8791
epoch 300 | loss 0.2476 | acc 0.8858
epoch 400 | loss 0.2385 | acc 0.8871
epoch 500 | loss 0.2272 | acc 0.8948
epoch 600 | loss 0.2052 | acc 0.9029
epoch 700 | loss 0.1973 | acc 0.9059
epoch 800 | loss 0.1987 | acc 0.9042
epoch 900 | loss 0.1775 | acc 0.9161
Scaled (SGD) test accuracy: 81.62%

--- Question 1.2: Raw Data with SGD ---
epoch 0 | loss 0.6835 | acc 0.7585
epoch 100 | loss 0.5532 | acc 0.7585
epoch 200 | loss 0.5528 | acc 0.7585
epoch 300 | loss 0.5528 | acc 0.7585
epoch 400 | loss 0.5528 | acc 0.7585
epoch 500 | loss 0.5528 | acc 0.7585
epoch 600 | loss 0.5528 | acc 0.7585
epoch 700 | loss 0.5528 | acc 0.7585
epoch 800 | loss 0.5528 | acc 0.7585
epoch 900 | loss 0.5528 | acc 0.7585
Raw (SGD) test accuracy: 75.50%

--- Question 1.2: Raw Data with Adam ---
epoch 0 | loss 7.5612 | acc 0.7808
epo